# STAC data prep — Country boundaries (GeoParquet)

Converts the Natural Earth 1:10m "Admin 0 – Countries" shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 258 country polygons with 168 attribute columns, EPSG:4326, no time dimension (~10 MB as shapefile).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `18_hydrolakes`). The layer is small, so no tiling or partitioning is needed.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas (2.28.5) has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_Country_bound.json`. It is incomplete (empty bbox, time, keywords, media type, author, citation, collection id), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\ne_10m_admin_0_countries.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 19_country_boundaries.ipynb -Variable ne_10m_admin_0_countries` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "ne_10m_admin_0_countries"
COLLECTION_ID = "country_boundaries"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_Country_bound.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: /workspace/africa_data/Data_Vivian/ne_10m_admin_0_countries.shp
Metadata template: /workspace/global-coastal-atlas/STAC/data/metadata/metadata_Country_bound.json
Output dir: /workspace/africa_data/Data_Vivian/stac/country_boundaries


### Read raw data

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio")

print("rows:", len(gdf), "| columns:", len(gdf.columns), "| crs:", gdf.crs)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("NE_ID unique:", gdf["NE_ID"].is_unique, "| null geometries:", int(gdf.geometry.isna().sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf[["NAME", "ISO_A3", "CONTINENT", "POP_EST"]].head()

rows: 258 | columns: 169 | crs: EPSG:4326
geometry types: {'MultiPolygon': 151, 'Polygon': 107}
NE_ID unique: True | null geometries: 0
total bounds: [-179.99999999999991, -89.99999999999994, 180.0, 83.63410065300008]


,NAME,ISO_A3,CONTINENT,POP_EST
0,Indonesia,IDN,Asia,270625568.0
1,Malaysia,MYS,Asia,31949777.0
2,Chile,CHL,South America,18952038.0
3,Bolivia,BOL,South America,11513100.0
4,Peru,PER,South America,32510453.0


### Check the CRS

The shapefile is in WGS84 (`GCS_WGS_1984`), the CRS the STAC catalog uses. A missing `.prj` is set to EPSG:4326; any other CRS is reprojected.

In [4]:
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())

CRS: EPSG:4326


### Make the metadata complete (dataset dependent)

Fill the fields that are empty in the template. Natural Earth has 168 attribute columns; the most used ones get an explicit description, the repeated families (`NAME_xx`, `ADM0_A3_xx`, `FCLASS_xx`) get a generated one, and the rest points to the Natural Earth documentation.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

known = {
    "featurecla": "Natural Earth feature class (Admin-0 country)",
    "scalerank": "Natural Earth scale rank (map scale at which the feature is shown)",
    "LABELRANK": "Label rank, used to prioritise labels",
    "SOVEREIGNT": "Name of the sovereign state",
    "SOV_A3": "Three-letter code of the sovereign state",
    "TYPE": "Type of country (e.g. Sovereign country, Dependency, Country)",
    "ADMIN": "Administrative unit name",
    "ADM0_A3": "Three-letter administrative unit code",
    "GEOUNIT": "Geographic unit name",
    "SUBUNIT": "Subunit name",
    "NAME": "Short country name",
    "NAME_LONG": "Long country name",
    "ABBREV": "Abbreviated country name",
    "POSTAL": "Postal abbreviation",
    "FORMAL_EN": "Formal English name",
    "FORMAL_FR": "Formal French name",
    "NAME_SORT": "Name used for sorting",
    "NAME_ALT": "Alternative name",
    "POP_EST": "Estimated population",
    "POP_RANK": "Population rank",
    "POP_YEAR": "Year of the population estimate",
    "GDP_MD": "Gross domestic product (million USD)",
    "GDP_YEAR": "Year of the GDP estimate",
    "ECONOMY": "Economy classification",
    "INCOME_GRP": "World Bank income group",
    "FIPS_10": "FIPS 10-4 country code",
    "ISO_A2": "ISO 3166-1 alpha-2 code",
    "ISO_A3": "ISO 3166-1 alpha-3 code",
    "ISO_N3": "ISO 3166-1 numeric code",
    "UN_A3": "United Nations numeric code",
    "WB_A2": "World Bank two-letter code",
    "WB_A3": "World Bank three-letter code",
    "CONTINENT": "Continent",
    "REGION_UN": "United Nations region",
    "SUBREGION": "United Nations subregion",
    "REGION_WB": "World Bank region",
    "LABEL_X": "Longitude of the label point",
    "LABEL_Y": "Latitude of the label point",
    "NE_ID": "Unique Natural Earth feature id",
    "WIKIDATAID": "Wikidata identifier",
    "geometry": "Country polygon (EPSG:4326)",
}


def describe(col: str) -> str:
    if col in known:
        return known[col]
    if col.startswith("NAME_") and col[5:].isalpha() and len(col) <= 8:
        return f"Country name in language '{col[5:].lower()}'"
    if col.startswith("ADM0_A3_"):
        return f"Administrative unit code as recognised by '{col[8:]}' (point-of-view variant)"
    if col.startswith("FCLASS_"):
        return f"Feature class as recognised by '{col[7:]}' (point-of-view variant)"
    return "Natural Earth attribute (see the Natural Earth Admin 0 – Countries documentation)"


table_columns = {col: describe(col) for col in gdf.columns}

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "CountryBoundaries",
    "INSTITUTION": "Natural Earth",
    "HISTORY": [
        "Natural Earth version 5.1.1, 1:10m Admin 0 - Countries, shapefile ne_10m_admin_0_countries.shp",
        "Converted to a single GeoParquet (zstd, EPSG:4326, rows sorted along a Hilbert curve, bbox covering column)",
        "Geometries and attributes are unchanged",
    ],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Release year of Natural Earth 5.1.1; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2022-01-01T00:00:00Z", ""],
    "KEYWORDS": ["countries", "administrative boundaries", "borders", "Natural Earth", "global"],
    "TAGS": ["countries", "boundaries", "polygons", "geoparquet"],
    "LONG_NAME": "Country boundaries of the world (1:10m)",
    "AUTHOR": "Natural Earth",
    "CITATION": (
        "Natural Earth (2022) Admin 0 - Countries, 1:10m, version 5.1.1. "
        "https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/"
    ),
    "COMMENT": (
        "Version 5.1.1, name: ne_10m_admin_0_countries.shp. Vector layer with one polygon per country (258 features). "
        "Natural Earth shows de facto boundaries; point-of-view variants are in the ADM0_A3_xx / FCLASS_xx columns. "
        "The temporal extent is the release year; the layer is static."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])
print("License in the template:", metadata["LICENSE"])

SPATIAL_EXTENT: [-180.0, -90.0, 180.0, 83.634101]
FEATURE_COUNT: 258
License in the template: Creative Commons Attribution 4.0


### Write data to GeoParquet

`zstd` compression, rows ordered along a Hilbert curve, and a `bbox` covering column for spatial filtering.

In [6]:
gdf = gdf.iloc[gdf.geometry.hilbert_distance().argsort()]

out_path = parquet_dir / f"{COLLECTION_ID}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB, shapefile parts {before_mb:.1f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: /workspace/africa_data/Data_Vivian/stac/country_boundaries/parquet/country_boundaries.parquet (5.9 MB, shapefile parts 9.7 MB)
Wrote metadata: /workspace/africa_data/Data_Vivian/stac/country_boundaries/metadata_country_boundaries.json


### Verify the output

Same rows, columns and values as the shapefile, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path).set_index("NE_ID").sort_index()
old = gdf.set_index("NE_ID").sort_index()
assert new.crs.to_epsg() == 4326
assert new.index.is_unique and new.index.equals(old.index)
assert list(new.columns.drop("bbox", errors="ignore")) == list(old.columns)

attrs = [c for c in old.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(old[attrs]))
assert new.geometry.geom_equals_exact(old.geometry, tolerance=0).all(), "Geometries changed"
print("OK: country_boundaries GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 1 | rows: 258


OK: country_boundaries GeoParquet and metadata are ready for the STAC builder notebook.
